In [26]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")

Main.BeamToyModel

In [17]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = lmax_in)
;

In [18]:
cs = ConvolutionSky()
fieldnames(typeof(cs))
cb = ConvolutionBeam()
fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
fieldnames(typeof(cc))

(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [19]:
nalm = Healpix.numberOfAlms(lmax_in, lmax_in)
almT = rand(ComplexF64, nalm)
almE = rand(ComplexF64, nalm)
almB = rand(ComplexF64, nalm)
cs.lmax = lmax_in
cs.alm = hcat(almT, almE, almB)
;

In [20]:
cb.lmax = lmax_in
#cb.mmax = 2
cb.blm = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
;

In [21]:
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
D_beam = spzeros(n_beam, n_beam)
n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
D_sky = spzeros(n_sky, n_beam)
;

In [46]:
nptg = 10
θ = pi/2
φ = pi/4
dθ = rand(nptg)*1e-2
dφ = rand(nptg)*1e-2
ψ = rand(nptg)*2pi

10-element Vector{Float64}:
 3.0319668582450645
 2.8579604880330485
 1.8007389084395478
 5.620650087204563
 2.9928288716597926
 2.4092023639571414
 0.8656216658003467
 4.598194343545822
 3.0201945166034823
 4.1493726899596295

In [47]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [48]:
@time A = local_effective_wignerD(cb, cc, alphas, betas, gammas)
@time B = global_wignerD(cc, φ, θ, 0.0)
@time C = B*A

  0.636323 seconds (27 allocations: 3.442 MiB)
  9.394543 seconds (33 allocations: 56.784 MiB)
  0.008238 seconds (7 allocations: 81.694 MiB)


9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46070 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [49]:
C

9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46070 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [50]:
@time D = local_effective_wignerD(cb, cc, φ.+dφ, θ .+dθ, ψ)

  0.830037 seconds (31 allocations: 3.443 MiB)


9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46070 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [51]:
maximum(abs.(C-D))

1.1370548028823726e-13

In [52]:
B[2:5,2:5]

4×4 SparseMatrixCSC{ComplexF64, Int64} with 10 stored entries:
  0.353553+0.353553im          0.5+0.5im  …              ⋅    
 -0.707107+0.0im       6.12323e-17-0.0im                 ⋅    
  0.353553-0.353553im         -0.5+0.5im                 ⋅    
           ⋅                       ⋅         1.53081e-17+0.25im

In [53]:
C[2:5,2:5]

4×4 SparseMatrixCSC{ComplexF64, Int64} with 10 stored entries:
  -0.16575-0.124417im     0.496925+0.503031im     …            ⋅    
  0.291196-0.0430047im  -0.0061442-1.47451e-17im               ⋅    
 -0.123765+0.168456im    -0.496925+0.503031im                  ⋅    
           ⋅                       ⋅                 0.0385146+0.0413469im

In [54]:
D[2:5,2:5]

4×4 SparseMatrixCSC{ComplexF64, Int64} with 10 stored entries:
  -0.16575-0.124417im     0.496925+0.503031im  …            ⋅    
  0.291196-0.0430047im  -0.0061442+0.0im                    ⋅    
 -0.123765+0.168456im    -0.496925+0.503031im               ⋅    
           ⋅                       ⋅              0.0385146+0.0413469im

In [13]:
cc.lstop

95

In [14]:
cc.lstop

95